# 04 Line Following With Stop Sign Action

This notebook launches the combined line-following and stop-sign action panel from `scripts/jetcar_contour_sign_panel.py`.

It reuses:

- `.jetcar_motor_calibration.json` from notebook `01`
- `.contour_follow_settings.json` for the contour detection sliders
- `models/traffic_sign_detector.pt` for the traffic-sign model

This is the stop-sign workflow: contour line following still handles steering, and a stable stop-sign detection triggers the hold action.


## Safety

Start on a stand, then move to the track only after camera, serial, YOLO, and test preset buttons all look correct. Keep `Stop Auto` or `Stop Motors` ready.

A stable `stop` detection will pause the rover automatically, so test the sign response carefully before running on the floor.


In [1]:
import glob
import os
import socket
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
video_devices = sorted(glob.glob('/dev/video*'))
serial_candidates = [p for p in ('/dev/ttyTHS1', '/dev/ttyTHS2') if os.path.exists(p)]
hostname = socket.gethostname()
ip_addresses = sorted({addr[-1][0] for addr in socket.getaddrinfo(hostname, None, family=socket.AF_INET)})
yolo_model_candidates = sorted(str(path.relative_to(project_root)) for path in project_root.rglob('*.pt'))

print('Project root:', project_root)
print('Video devices:', video_devices or ['none found'])
print('Serial candidates:', serial_candidates or ['none found'])
print('Calibration file exists:', (project_root / '.jetcar_motor_calibration.json').exists())
print('Contour settings file exists:', (project_root / '.contour_follow_settings.json').exists())
print('YOLO model files:', yolo_model_candidates or ['none found in this repo'])
print('Host IPv4:', ip_addresses or ['127.0.0.1'])


Project root: /home/orin/JetCar
Video devices: ['/dev/video0']
Serial candidates: ['/dev/ttyTHS1', '/dev/ttyTHS2']
Calibration file exists: True
Contour settings file exists: False
YOLO model files: ['models/traffic_sign_detector.pt', 'yolo11n.pt']
Host IPv4: ['127.0.1.1']


In [2]:
camera_source = 'csi'
host = '0.0.0.0'
http_port = 8765
serial_port = serial_candidates[0] if serial_candidates else '/dev/ttyTHS1'
baudrate = 115200
sensor_id = 0
device_index = 0
frame_width = 1280
frame_height = 720
warmup_frames = 12

default_stop_model = project_root / 'models' / 'traffic_sign_detector.pt'
yolo_model = str(default_stop_model)
yolo_conf = 0.25
yolo_imgsz = 640
yolo_every_n = 1
yolo_device = ''
yolo_max_det = 20
sign_vote_window = 5
sign_vote_threshold = 3
sign_min_conf = 0.35

print('Edit the camera, serial, or stop-sign values here if needed, then run the next cells.')
print('Current stop-sign model:', yolo_model)


Edit the camera, serial, or stop-sign values here if needed, then run the next cells.
Current stop-sign model: /home/orin/JetCar/models/traffic_sign_detector.pt


In [3]:
import shlex
import subprocess
import sys
import time
from IPython.display import HTML, display

if '_contour_sign_process' not in globals():
    _contour_sign_process = None

panel_log_path = project_root / '.jetcar_contour_sign_panel.notebook.log'

def panel_python() -> str:
    candidate = project_root / '.venv' / 'bin' / 'python'
    return str(candidate) if candidate.exists() else sys.executable


def panel_command() -> list[str]:
    return [
        panel_python(),
        str(project_root / 'scripts' / 'jetcar_contour_sign_panel.py'),
        '--host', host,
        '--http-port', str(http_port),
        '--camera-source', camera_source,
        '--sensor-id', str(sensor_id),
        '--device-index', str(device_index),
        '--width', str(frame_width),
        '--height', str(frame_height),
        '--warmup-frames', str(warmup_frames),
        '--port', serial_port,
        '--baudrate', str(baudrate),
        '--yolo-model', str(yolo_model),
        '--yolo-conf', str(yolo_conf),
        '--yolo-imgsz', str(yolo_imgsz),
        '--yolo-every-n', str(yolo_every_n),
        '--yolo-device', str(yolo_device),
        '--yolo-max-det', str(yolo_max_det),
        '--sign-vote-window', str(sign_vote_window),
        '--sign-vote-threshold', str(sign_vote_threshold),
        '--sign-min-conf', str(sign_min_conf),
    ]


def stop_contour_sign_panel() -> None:
    global _contour_sign_process
    if _contour_sign_process is None:
        return
    if _contour_sign_process.poll() is None:
        _contour_sign_process.terminate()
        try:
            _contour_sign_process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            _contour_sign_process.kill()
            _contour_sign_process.wait(timeout=3)
    _contour_sign_process = None


def browser_url() -> str:
    browser_host = '127.0.0.1' if host == '0.0.0.0' else host
    return f'http://{browser_host}:{http_port}'

print('Equivalent terminal command:')
print(shlex.join(panel_command()))


Equivalent terminal command:
/home/orin/JetCar/.venv/bin/python /home/orin/JetCar/scripts/jetcar_contour_sign_panel.py --host 0.0.0.0 --http-port 8765 --camera-source csi --sensor-id 0 --device-index 0 --width 1280 --height 720 --warmup-frames 12 --port /dev/ttyTHS1 --baudrate 115200 --yolo-model /home/orin/JetCar/models/traffic_sign_detector.pt --yolo-conf 0.25 --yolo-imgsz 640 --yolo-every-n 1 --yolo-device '' --yolo-max-det 20 --sign-vote-window 5 --sign-vote-threshold 3 --sign-min-conf 0.35


In [4]:
stop_contour_sign_panel()
cmd = panel_command()
print('Starting line following + stop-sign action panel...')
print(shlex.join(cmd))
with open(panel_log_path, 'w') as log_file:
    _contour_sign_process = subprocess.Popen(
        cmd,
        cwd=project_root,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
time.sleep(2.0)
if _contour_sign_process.poll() is not None:
    raise RuntimeError(f'line following + stop-sign action panel exited early; check {panel_log_path}')
print('Line following + stop-sign action panel running at:', browser_url())
print('Stop-sign model:', yolo_model)
if ip_addresses:
    print('LAN example:', f'http://{ip_addresses[0]}:{http_port}')
display(HTML(
    f'<p><a href="{browser_url()}" target="_blank">Open the line following + stop-sign action panel in a new tab</a></p>'
    f'<iframe src="{browser_url()}" width="100%" height="900" style="border:1px solid #ccc; border-radius:12px;"></iframe>'
))


Starting line following + stop-sign action panel...
/home/orin/JetCar/.venv/bin/python /home/orin/JetCar/scripts/jetcar_contour_sign_panel.py --host 0.0.0.0 --http-port 8765 --camera-source csi --sensor-id 0 --device-index 0 --width 1280 --height 720 --warmup-frames 12 --port /dev/ttyTHS1 --baudrate 115200 --yolo-model /home/orin/JetCar/models/traffic_sign_detector.pt --yolo-conf 0.25 --yolo-imgsz 640 --yolo-every-n 1 --yolo-device '' --yolo-max-det 20 --sign-vote-window 5 --sign-vote-threshold 3 --sign-min-conf 0.35
Line following + stop-sign action panel running at: http://127.0.0.1:8765
Stop-sign model: /home/orin/JetCar/models/traffic_sign_detector.pt
LAN example: http://127.0.1.1:8765


In [5]:
print('Last line following + stop-sign action panel log lines:')
if panel_log_path.exists():
    print('\n'.join(panel_log_path.read_text().splitlines()[-60:]))
else:
    print('No log file yet.')


Last line following + stop-sign action panel log lines:



In [ ]:
# stop_contour_sign_panel()
# print('Line following + stop-sign action panel stopped.')


## What To Do In The Panel

1. make sure `Camera`, `Serial`, and `YOLO` become ready
2. confirm the RGB image is live and the traffic-sign model sees your stop sign
3. confirm the grayscale ROI, mask, and contour overlay still track the line well
4. use the preset test buttons to confirm the saved calibration still feels correct
5. watch `Stable Sign` and `Detection Metrics` before enabling auto drive
6. place the rover on the line and press `Auto Start`
7. show a stop sign to the camera and confirm the rover pauses automatically
8. remove the sign and confirm the rover can continue after the hold finishes
9. press `Stop Auto` or `Stop Motors` immediately if behavior looks wrong

This notebook matches the working command that uses `models/traffic_sign_detector.pt` with `--yolo-conf 0.25`.
